# 08 — Avaliação final, limites e respostas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/projeto-plumas/blob/main/08_avaliacao/08_avaliacao.ipynb)

**Objetivo:** reunir os resultados de todas as etapas, responder às perguntas de pesquisa escolhidas e deixar explícitos os limites do método.

**Escopo da professora:** seções 2 (perguntas), 1 (avaliar desempenho e limites) e 10 (reprodutibilidade).

**Lê:** tabelas das etapas 03 a 07 e `runs.jsonl`.

**Grava:** `tabelas/` e `figuras/` neste diretório do repositório (pequenos, versionados) para o relatório.

**GPU:** não


In [ ]:
# === Preparo (igual em todos os notebooks) ===
import os, sys, subprocess

REPO = "USUARIO/projeto-plumas"      # troque pelo seu (veja README)
REPO_DIR = "/content/projeto-plumas"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    try:                                   # repositório privado: secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        _tok = None
    _url = f"https://{_tok}@github.com/{REPO}.git" if _tok else f"https://github.com/{REPO}.git"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True)
    else:
        _r = subprocess.run(["git", "clone", "-q", _url, REPO_DIR], capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("Falha no clone. Repositório privado? Configure o secret GITHUB_TOKEN (veja README).")
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:                                      # execução local: sobe até a raiz do repo
    _p = os.getcwd()
    while not os.path.exists(os.path.join(_p, "config.yaml")) and os.path.dirname(_p) != _p:
        _p = os.path.dirname(_p)
    os.chdir(_p)
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from pathlib import Path
from plumas.config import load_config, stage_dir, set_seed, log_experiment
cfg = load_config()
set_seed(cfg["seed"])
STAGE = "08_avaliacao"
D = stage_dir(cfg, STAGE)                  # pasta desta etapa no Drive
print("Etapa:", STAGE, "| dados em:", D)


In [ ]:
import pandas as pd, json
import matplotlib.pyplot as plt
from IPython.display import display

out_tab = Path(cfg["_repo_root"]) / STAGE / "tabelas"; out_tab.mkdir(exist_ok=True)
out_fig = Path(cfg["_repo_root"]) / STAGE / "figuras"; out_fig.mkdir(exist_ok=True)

def carregar(stage, nome):
    p = stage_dir(cfg, stage) / nome
    return pd.read_csv(p) if p.exists() else None

det = carregar("03_yolo_treino", "metricas_deteccao.csv")
seg = carregar("04_sam_atributos", "segmentacao_metricas.csv")
tr  = carregar("05_tracking", "trajetoria.csv")
sem = carregar("05_tracking", "sem_deteccao.csv")
ven = carregar("06_era5", "vento_vs_deslocamento.csv")
sr  = carregar("07_super_resolucao", "metricas_sr.csv")
for nome, t in [("detecção", det), ("segmentação", seg), ("trajetória", tr), ("sem detecção", sem), ("vento", ven), ("super-resolução", sr)]:
    print(nome, ":", "ok" if t is not None else "FALTA")

## Tabela 1: detecção (pergunta 1)
Variantes A (zero-shot), B (só públicos) e C (públicos + próprios), no mesmo teste.

In [ ]:
if det is not None:
    t1 = det[["variante", "imgsz", "precision", "recall", "mAP50", "mAP50_95"]].round(3)
    display(t1); t1.to_csv(out_tab / "tabela1_deteccao.csv", index=False)
    ax = t1.set_index("variante")[["mAP50", "mAP50_95"]].plot.bar(figsize=(7, 4), rot=20); ax.set_ylim(0, 1); ax.set_title("Detecção no teste")
    plt.tight_layout(); plt.savefig(out_fig / "fig1_deteccao.png", dpi=150); plt.show()

## Tabela 2: segmentação (pergunta 2)
IoU e Dice das máscaras automáticas do SAM contra referência manual. Reporte média, desvio e os piores casos.

In [ ]:
if seg is not None:
    t2 = seg[["iou", "dice"]].agg(["count", "mean", "std", "min", "max"]).round(3)
    display(t2); t2.to_csv(out_tab / "tabela2_segmentacao.csv")
    display(seg.sort_values("iou").head(5))      # piores quadros: inspecione e discuta

## Tabela 3: comportamento temporal (pergunta 3)
Variação de área, orientação e centroide entre quadros consecutivos, e cobertura das detecções.

In [ ]:
if tr is not None:
    t3 = tr[["dx_px", "dy_px", "speed_px_s", "d_area_px", "d_orient_deg"]].describe().round(2)
    display(t3); t3.to_csv(out_tab / "tabela3_temporal.csv")
    if sem is not None:
        print(f"Quadros sem detecção: {len(sem)} de {len(sem) + tr['frame'].nunique()} ({100 * len(sem) / (len(sem) + tr['frame'].nunique()):.1f}%)")

## Tabela 4: vento vs. deslocamento (pergunta 4)
Contagem de passos em que o sinal do deslocamento lateral concorda com o vento lateral projetado.
**Compatibilidade não prova causalidade.**

In [ ]:
if ven is not None:
    t4 = ven["concordancia"].value_counts().rename_axis("sinal").reset_index(name="passos")
    display(t4); t4.to_csv(out_tab / "tabela4_vento.csv", index=False)

## Tabela 5: super-resolução (pergunta 5)
`esrgan_SR` deve ser comparada com `bicubic_SR` (controle), não só com o original.

In [ ]:
if sr is not None:
    t5 = sr.round(3); display(t5); t5.to_csv(out_tab / "tabela5_sr.csv", index=False)

## Registro de experimentos (reprodutibilidade)
`runs.jsonl` guarda cada etapa com data, commit do código, parâmetros e decisões.

In [ ]:
runs_path = Path(cfg["drive_root"]) / "runs.jsonl"
if runs_path.exists():
    runs = pd.DataFrame([json.loads(l) for l in runs_path.read_text(encoding="utf-8").splitlines()])
    display(runs[["timestamp_utc", "stage", "git_commit"] + [c for c in ("variante", "decisao") if c in runs.columns]].tail(30))
    runs.to_csv(out_tab / "registro_experimentos.csv", index=False)

## Respostas às perguntas de pesquisa (preencher com a dupla)
Para cada pergunta escolhida: **Pergunta → Evidência (tabela/figura) → Conclusão → Limite**.

1. *Detecção em condições diferentes?* …
2. *Segmentação consistente com a forma visível?* …
3. *Como área, orientação e centroide variam?* …
4. *Direção compatível com o vento?* …
5. *Super-resolução ajuda sem introduzir erros?* …

## Limites do método (discutir no relatório)
- [ ] Medidas na imagem são **aparentes** (pixels/proporções); sem calibração, não há metros nem m/s.
- [ ] Deslocamento aparente, vento do ERA5 e trajetória prevista por modelo físico são **coisas diferentes**; aqui não houve previsão de trajetória.
- [ ] ERA5: grade de ≈ 0,25° e vento de superfície, que pode não descrever o microambiente nem a altura da pluma; a perspectiva da câmera limita a comparação a movimento lateral.
- [ ] Compatibilidade de direção ≠ causalidade.
- [ ] Tamanho e diversidade do vídeo: quantas sequências, locais, condições de luz? Quanto o resultado depende de um único vídeo?
- [ ] Confusões observadas (nuvem, neblina, vapor) e condições em que o detector falha.
- [ ] Máscaras do SAM: bordas difusas e baixo contraste; revisão manual de uma amostra.
- [ ] Super-resolução: possíveis artefatos e comparação com o controle bicúbico.